# RSM from Bruker .brml: source left, detector right

$$Q_x = R\,[\cos(2\theta-\omega) - \cos\omega] \qquad Q_z = R\,[\sin\omega + \sin(2\theta-\omega)] \qquad R = 2\pi/\lambda$$

ω is the angle of incidence (from the source side) and 2θ − ω is the exit angle (towards the detector).
* **Grazing incidence** (ω < θ): Qx < 0
* **Grazing exit** (ω > θ): Qx > 0

The equations apply to any asymmetric reflection measured in this coplanar setup.
xrayutilities with `QConversion(['y-','x+','z-'], ['y-'], [1,0,0])` uses the same sign convention, so both methods give identical maps.

The .brml supplies everything the conversion needs:
* ω and 2θ for every step
* the LynxEye calibration: channel *i* sits at 2θ + `ZeroOffset` + *i*·`Resolution`
* the frames: n_steps × n_channels, stored as float64

No pixel size or detector distance is needed.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import xrayutilities as xu
from matplotlib.colors import LogNorm

## 1. Reader

In [ ]:
import os, zipfile
import xml.etree.ElementTree as ET
import numpy as np

def read_bruker_brml(path, raw_index=0):
    """Read a Bruker .brml file (zip of XML + binary 1D-detector frames).

    Works for any coplanar omega/2theta-type scan (asymmetric RSM, coupled scan with
    offset, rocking curve, detector scan) recorded with the LynxEye in 1D mode.
    Returns a dict with:
      omega, twotheta : (n_steps,) goniometer angles per step [deg] (Bruker: omega = incidence angle)
      delta_2theta    : (n_chan,) 2theta of each channel relative to the detector centre [deg]
      counts          : (n_steps, n_chan)
      chi, phi, wavelength, resolution, zero_offset, scan_name, name
    """
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        root = ET.fromstring(z.read(f"Experiment0/RawData{raw_index}.xml"))
        route = root.find(".//DataRoute")

        # --- per-step values: <Datum> is a comma-separated vector; RawDataViews say which
        # column is which. Collect every axis column, preferring ScanAxes over RandomAccess.
        datums = np.array([[float(v) for v in d.text.split(",")] for d in route.iter("Datum")])
        columns = {}
        for view in route.findall(".//DataViews/RawDataView"):
            varying = view.find("Varying")
            if varying is None:
                continue
            start = int(view.get("Start"))
            for k, fd in enumerate(varying.findall("FieldDefinitions")):
                axis = fd.get("AxisId")
                if axis and (axis not in columns or varying.get("LogicName") == "ScanAxes"):
                    columns[axis] = start + k

        # --- fixed drive positions (used when an axis is not scanned, e.g. 2theta in a rocking curve)
        fixed = {d.get("LogicName"): float(d.find("Position").get("Value"))
                 for d in root.findall(".//FixedInformation/Drives/InfoData")
                 if d.find("Position") is not None}

        def axis_values(axis):
            if axis in columns:
                return datums[:, columns[axis]]
            if axis in fixed:
                return np.full(len(datums), fixed[axis])
            raise KeyError(f"axis {axis!r} found neither in scan data nor in fixed drives")

        omega, twotheta = axis_values("Theta"), axis_values("TwoTheta")

        # --- detector calibration: channel i sits at 2theta + ZeroOffset + i * Resolution
        scale = route.find(".//ScaleAxes/ScaleAxisInfo")
        zero_offset = float(scale.find("Start").text)
        resolution = float(scale.find("Increment").text)
        n_chan = int(route.find(".//RawDataView[@MeasuredDataIndex='0']/Recording/Size/X").text)
        delta_2theta = zero_offset + resolution * np.arange(n_chan)

        # --- frames: n_steps x n_chan little-endian float64
        link = route.find(".//ScanDataExternalLink")
        bin_name = next(n for n in names if n.endswith(link.get("Name")))
        counts = np.frombuffer(z.read(bin_name), dtype="<f8").reshape(len(omega), n_chan)

        wl = root.find(".//WaveLength")
        scan = route.find("ScanInformation")

    return dict(omega=omega, twotheta=twotheta, delta_2theta=delta_2theta, counts=counts,
                chi=fixed.get("Chi", 0.0), phi=fixed.get("Phi", 0.0),
                wavelength=float(wl.get("Value")) if wl is not None else 1.5406,
                resolution=resolution, zero_offset=zero_offset,
                scan_name=scan.get("VisibleName") if scan is not None else "",
                name=os.path.basename(path))


## 2. Conversion: equations (coplanar) and xrayutilities (includes χ)

In [ ]:
def angles_to_q(omega, twotheta, delta_2theta, wavelength):
    """Source left, detector right (figure convention), every (step, channel) pixel:
        Qx = R [cos(2theta - omega) - cos(omega)]
        Qz = R [sin(omega) + sin(2theta - omega)],   R = 2 pi / lambda
    Each channel sees 2theta_i = 2theta + delta_i; omega is common to all channels.
    Qx < 0 for grazing incidence (omega < theta), Qx > 0 for grazing exit (omega > theta)."""
    R = 2 * np.pi / wavelength
    w = np.deg2rad(np.asarray(omega, float))[:, None]
    tt = np.deg2rad(np.asarray(twotheta, float)[:, None] + np.asarray(delta_2theta, float)[None, :])
    w = np.broadcast_to(w, tt.shape)
    Qx = R * (np.cos(tt - w) - np.cos(w))
    Qz = R * (np.sin(w) + np.sin(tt - w))
    return Qx, Qz


In [ ]:
def make_qconv_from_brml(d):
    """D8 geometry as in gao_sto03.1 (omega 'y-', chi 'x+', phi 'z-'; detector 'y-'),
    which gives the same sign convention as the figure: source left, detector right.
    The detector channels use Bruker's calibration from the .brml instead of pixel size/distance."""
    qconv = xu.QConversion(['y-', 'x+', 'z-'], ['y-'], [1, 0, 0], wl=d["wavelength"])
    qconv.init_linear('z+', -d["zero_offset"] / d["resolution"], len(d["delta_2theta"]),
                      chpdeg=1.0 / d["resolution"])
    return qconv

def brml_to_q(path, method="xu", use_chi=True):
    """Convert one .brml file to Q.  method="xu" -> xrayutilities (includes chi tilt if use_chi),
    method="eq" -> the coplanar equations above.  Returns Qx, Qy, Qz, I, d (2D arrays)."""
    d = read_bruker_brml(path)
    if method == "eq":
        Qx, Qz = angles_to_q(d["omega"], d["twotheta"], d["delta_2theta"], d["wavelength"])
        Qy = np.zeros_like(Qx)
    else:
        n = len(d["omega"])
        chi = np.full(n, d["chi"] if use_chi else 0.0)
        Qx, Qy, Qz = make_qconv_from_brml(d).linear(d["omega"], chi, np.full(n, d["phi"]), d["twotheta"])
    return Qx, Qy, Qz, d["counts"], d


## 3. Expected positions for any (hkl): which ω and 2θ to measure, and where the peak should appear

In [ ]:
def expected_reflection(hkl, a, c=None, b=None, wavelength=1.5406):
    """Expected position of an (hkl) reflection of an orthogonal lattice (cubic / tetragonal /
    orthorhombic, surface normal = c) in the source-left / detector-right geometry.
    Returns both settings that reach it with the same detector arm:
      'grazing_incidence' (omega < theta, Qx < 0) and 'grazing_exit' (omega > theta, Qx > 0)."""
    h, k, l = hkl
    b = a if b is None else b
    c = a if c is None else c
    q_ip = 2 * np.pi * np.hypot(h / a, k / b)
    q_z = 2 * np.pi * l / c
    q = np.hypot(q_ip, q_z)
    theta = np.degrees(np.arcsin(q * wavelength / (4 * np.pi)))
    psi = np.degrees(np.arctan2(q_ip, q_z))  # tilt of Q away from the surface normal
    return {
        "2theta": 2 * theta, "psi": psi, "Qz": q_z,
        # accessible = beam enters and leaves above the surface (omega > 0 and 2theta - omega > 0)
        "grazing_incidence": dict(omega=theta - psi, Qx=-q_ip, accessible=theta - psi > 0),
        "grazing_exit": dict(omega=theta + psi, Qx=q_ip, accessible=psi < theta),
    }

def geometry_of_scan(d):
    """'grazing_exit' if omega > theta over the scan, else 'grazing_incidence'."""
    return "grazing_exit" if np.median(d["omega"] - d["twotheta"] / 2) > 0 else "grazing_incidence"


In [ ]:
# example: substrate and other asymmetric reflections for SrTiO3 (a = 3.905 Å)
for hkl in [(0, 1, 3), (1, 0, 3), (1, 1, 3), (-1, 0, 3), (0, 2, 4), (1, 0, 2)]:
    e = expected_reflection(hkl, a=3.905)
    gi, ge = e["grazing_incidence"], e["grazing_exit"]
    flag = lambda g: "" if g["accessible"] else " (not reachable)"
    print(f"{hkl}: 2θ = {e['2theta']:7.3f}°   GI: ω = {gi['omega']:7.3f}°{flag(gi)}, Qx = {gi['Qx']:+.4f}"
          f"   GE: ω = {ge['omega']:7.3f}°{flag(ge)}, Qx = {ge['Qx']:+.4f}   Qz = {e['Qz']:.4f} Å⁻¹")

## 4. Plotting

In [ ]:
def plot_rsm(Qh, Qz, I, title, horizontal_label="Qx", markers=None):
    """Qh, Qz, I are 2D (n_steps, n_channels) arrays at native resolution.
    markers: optional list of (Qh, Qz, label) to overlay, e.g. bulk substrate position."""
    I_plot = np.where(I > 0, I, np.nan)
    fig, ax = plt.subplots(figsize=(5, 6), dpi=200)
    ax.set_facecolor("black")
    im = ax.pcolormesh(Qh, Qz, I_plot, cmap="inferno",
                       norm=LogNorm(vmin=1, vmax=np.nanmax(I)), shading="auto")
    for qh, qz, label in (markers or []):
        ax.plot(qh, qz, "+", color="cyan", ms=12, mew=1)
        ax.annotate(label, (qh, qz), xytext=(6, 6), textcoords="offset points", color="cyan", fontsize=7)
    ax.set_xlabel(f"{horizontal_label} / Å⁻¹"); ax.set_ylabel("Qz / Å⁻¹")
    ax.set_title(title, fontsize=7)
    fig.colorbar(im, ax=ax, label="Intensity (counts)")
    fig.tight_layout()
    plt.show()

def plot_brml_file(path, method="xu", use_chi=True, show_qx=True, show_qy=False,
                   reference=None):
    """reference: optional dict(hkl=(0,1,3), a=3.905, c=None, label="STO") -> marks the expected
    bulk position for the geometry (grazing incidence/exit) this scan was measured in."""
    Qx, Qy, Qz, I, d = brml_to_q(path, method=method, use_chi=use_chi)
    geom = geometry_of_scan(d)
    markers = None
    if reference is not None:
        ref = expected_reflection(reference["hkl"], reference["a"], reference.get("c"),
                                  wavelength=d["wavelength"])
        label = f'{reference.get("label", "")} {"".join(map(str, reference["hkl"]))}'.strip()
        markers = [(ref[geom]["Qx"], ref["Qz"], label)]
    title = f'{d["name"]}\n{d["scan_name"]}, {geom.replace("_", " ")}, {method}'
    if show_qx:
        plot_rsm(Qx, Qz, I, title, "Qx", markers)
    if show_qy:
        plot_rsm(Qy, Qz, I, title, "Qy")
    return Qx, Qy, Qz, d

def plot_all_brml_in_dir(dir_path, **kwargs):
    files = sorted(f for f in os.listdir(dir_path) if f.lower().endswith(".brml"))
    if not files:
        print(f"no .brml files found in {dir_path}")
    for fname in files:
        print(f"--- {fname} ---")
        plot_brml_file(os.path.join(dir_path, fname), **kwargs)


In [ ]:
dir_1 = r"C:\Users\j.bantol\Documents\Data\RSM\PGI-7\GAO_STO03.1\2026-10-15"
dir_2 = r"C:\Users\j.bantol\Documents\Data\RSM\PGI-7\GAO_STO03.1\2026-10-16"
dir_3 = r"C:\Users\j.bantol\Documents\Data\RSM\PGI-7\GAO_STO03.1\2026-10-29"

STO = dict(a=3.905, label="STO")

In [ ]:
%matplotlib inline
plt.close("all")
path = os.path.join(dir_1, "STO_013p_56.3284°-58.3484°_0.002°_3s_om_off_-36.7.brml")
Qx, Qy, Qz, d = plot_brml_file(path, method="xu", reference=dict(STO, hkl=(0, 1, 3)))

i, j = np.unravel_index(d["counts"].argmax(), d["counts"].shape)
print(f"brightest pixel: Qx = {Qx[i, j]:+.4f}, Qz = {Qz[i, j]:.4f} Å⁻¹  ({geometry_of_scan(d)})")

# cross-check: xrayutilities (chi = 0) vs the equations
Qx_xu, _, Qz_xu, _, _ = brml_to_q(path, method="xu", use_chi=False)
Qx_eq, _, Qz_eq, _, _ = brml_to_q(path, method="eq")
print("max |ΔQx| =", np.abs(Qx_xu - Qx_eq).max(), "  max |ΔQz| =", np.abs(Qz_xu - Qz_eq).max())

In [ ]:
%matplotlib inline
plt.close("all")
# every .brml in the folder; the marker is placed for the geometry each scan was measured in
plot_all_brml_in_dir(dir_1, method="xu", reference=dict(STO, hkl=(0, 1, 3)))